# Week 2: Retail Sales Data Inspection & Integrity Audit
**Project:** Retail Sales Forecasting & Inventory Planning Using Python  
**Internship:** YuvaIntern Virtual Data Science Explorer  
**Dataset:** Walmart Weekly Store Sales (45 Stores, Feb 2010 – Oct 2012)

---
### Objectives
1. Load and inspect the historical retail sales dataset.
2. Verify schema integrity, data types, nulls, and duplicates.
3. Validate the date coverage and time-series balance across stores.
4. Audit distribution statistics and conduct retail outlier diagnostics.
5. Document data cleaning and modeling decisions.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Setup root path for modular imports
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_cleaning import load_raw_data, inspect_data, detect_outliers
print('Environment initialized successfully.')

## 1. Load Raw Dataset
Load the benchmark dataset from `data/raw/walmart_sales.csv`.

In [ ]:
raw_df = load_raw_data(project_root / 'data' / 'raw' / 'walmart_sales.csv')
print(f'Dataset Shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
raw_df.head(5)

## 2. Schema, Nulls & Duplicates Inspection
Inspect structural validity, null values across all features, and duplicate rows.

In [ ]:
inspection = inspect_data(raw_df)
print(f'Total Missing Cells: {inspection["total_nulls"]}')
print(f'Duplicate Rows:      {inspection["duplicate_rows"]}')
print(f'Unique Outlets:      {inspection["unique_stores"]}')

schema_audit = pd.DataFrame({
    'Data_Type': raw_df.dtypes,
    'Null_Count': raw_df.isnull().sum(),
    'Null_Pct (%)': (raw_df.isnull().sum() / len(raw_df)) * 100,
    'Unique_Count': raw_df.nunique()
})
schema_audit

## 3. Date Span & Temporal Regularity
Check the temporal granularity and chronological coverage.

In [ ]:
dates_parsed = pd.to_datetime(raw_df['Date'], format='%d-%m-%Y')
print('Temporal Range:')
print(f'  - Start Date: {dates_parsed.min().strftime("%A, %B %d, %Y")}')
print(f'  - End Date:   {dates_parsed.max().strftime("%A, %B %d, %Y")}')
print(f'  - Total Unique Weekly Steps: {dates_parsed.nunique()}')
print(f'  - Records per Store: {len(raw_df) / raw_df["Store"].nunique():.0f}')

## 4. Target Variable Distribution (`Weekly_Sales`)
Calculate five-number summary and dispersion metrics.

In [ ]:
sales_desc = raw_df['Weekly_Sales'].describe()
formatted_desc = pd.DataFrame({
    'Metric': sales_desc.index,
    'Weekly_Sales_USD': sales_desc.values
})
formatted_desc['Formatted'] = formatted_desc['Weekly_Sales_USD'].apply(lambda x: f'${x:,.2f}')
formatted_desc

## 5. Outlier Diagnostics & Domain Decision
> **Domain Knowledge for Internship Evaluation:**  
> In standard tabular problems, rows outside $Q_3 + 1.5 \times IQR$ are often clipped or dropped. In retail forecasting, holiday surges (Thanksgiving/Black Friday and pre-Christmas) represent genuine high-revenue spikes. Dropping them results in severe under-forecasting and inventory stockouts. We preserve them and capture their seasonality with calendar and holiday indicators.

In [ ]:
outliers, metrics = detect_outliers(raw_df)
print(f'Upper Bound (Q3 + 1.5 * IQR): ${metrics["upper_bound"]:,.2f}')
print(f'Identified Outliers: {metrics["outlier_count"]} records ({metrics["outlier_percentage"]:.2f}% of data)')

# Display top outliers
outliers.sort_values('Weekly_Sales', ascending=False).head(10)[['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag']]

## 6. Summary of Data Inspection
| Inspection Item | Finding | Action / Conclusion |
| :--- | :--- | :--- |
| Total Rows / Columns | 6,435 rows × 8 columns | Confirmed balanced panel dataset. |
| Missing Values | 0 nulls across all fields | No imputation required. |
| Duplicate Records | 0 duplicates | Data is clean and unique. |
| Store Count | 45 stores (143 weeks each) | Balanced across all entities. |
| Outlier Policy | 34 rows above upper bound | Preserved as valid holiday demand surges. |